# 05 — Comparacao dos datasets

Pergunta: **quais combinacoes sao metodologicamente defensaveis?** Apresentamos evidencias sem escolher automaticamente uma estrategia.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, ensure_output_directories
from src.data.make_dataset import build_interim_datasets, load_interim_datasets

cfg = load_config()
ensure_output_directories(cfg)
if not list(Path(cfg["data"]["interim"]).glob("*.parquet")):
    build_interim_datasets()
datasets = {str(frame["dataset"].iloc[0]): frame for frame in load_interim_datasets().values()}
list(datasets)


In [ ]:
from src.analysis.profiling import dataset_overview, missingness_table, label_distribution
from src.analysis.duplicates import duplicate_summary, cross_dataset_duplicates
from src.analysis.text_metrics import add_text_features, summarize_text_features

overview = dataset_overview(datasets)
missing = pd.concat([missingness_table(frame, name) for name, frame in datasets.items()], ignore_index=True)
labels = pd.concat([label_distribution(frame, name) for name, frame in datasets.items()], ignore_index=True)
duplicates = pd.concat([duplicate_summary(frame, name) for name, frame in datasets.items()], ignore_index=True)
text_summary = pd.concat([summarize_text_features(add_text_features(frame), name) for name, frame in datasets.items()], ignore_index=True)
display(overview); display(labels); display(duplicates); display(text_summary)

## Cobertura, fontes e dominios

In [ ]:
pd.DataFrame([{"dataset": name, "unique_domains": frame["domain"].replace("", pd.NA).nunique(), "unique_authors": frame["author"].replace("", pd.NA).nunique(), "valid_dates": frame["date_parsed"].notna().sum(), "missing_text_percent": 100 * frame["text"].fillna("").str.strip().eq("").mean()} for name, frame in datasets.items()])

## A. Usar separadamente

Preserva tarefa, label e coleta. Favorece diagnostico intra-dominio, mas nao mede transferencia.

## B. Combinar subconjuntos

Somente apos alinhar unidade textual/taxonomia e criar grupos de duplicidade. `other` nao deve ser binarizado por conveniencia; datasets sem texto nao equivalem a classificacao textual.

## C. Treino e teste externo

Mede generalizacao entre fontes. Antes, remover toda sobreposicao e confirmar equivalencia de tarefa e label.

In [ ]:
cross = cross_dataset_duplicates(datasets)
cross.groupby(["datasets", "match_type", "label_conflict"]).size().rename("groups").reset_index()

## Decisao pendente

Revisar o relatorio, conflitos e quase duplicatas. A decisao deve declarar unidade de predicao, taxonomia, deduplicacao, split e populacao-alvo.